# GLiNER2 Jetson API — Worked Examples

Runnable client examples for the self-hosted GLiNER2 service on `jarvita-agx`
(NVIDIA Jetson AGX Orin 64GB).

The service exposes **12 routes**. Everything below runs against the live
deployment; nothing here is a mock.

Covered here:

1. Setup and connectivity
2. `GET /health`, `GET /health/deep`, `GET /version`
3. `POST /extract_entities`
4. `POST /classify_text`
5. `POST /extract_structured`
6. `POST /extract_multitask`
7. Error semantics (400 / 413 / 501 / 503 / 504)
8. Throughput: sequential vs concurrent
9. Inference options — `threshold`, `include_confidence`, `include_spans`,
   `max_len`, `overlap_policy`
10. Relation extraction — `POST /extract_relations` and
    `schema_config.relations`
11. The four batch routes
12. Batched vs sequential, measured on your deployment

Five routes need a GLiNER2.5 `boundary` checkpoint — `/extract_relations` and
all four `*_batch` routes. Sections 10-12 detect this and skip cleanly on a span
model rather than failing.

Companion docs: [`index.md`](index.md), [`api.md`](api.md),
[`runbook.md`](runbook.md), [`wiki.md`](wiki.md).

Only dependency is `requests`:

```bash
pip install requests
```


## 1. Setup

`BASE_URL` is read from the `GLINER_BASE_URL` environment variable so the same
notebook works against the Jetson, a local dev server, or a port-forward. Change
the fallback if your deployment differs.

| Deployment | Base URL |
|---|---|
| `jarvita-agx` on the LAN | `http://192.168.1.177:8013` |
| Container on the local host | `http://localhost:8013` |
| `make run` / `make dev` locally | `http://localhost:8125` |

In [ ]:
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor

import requests

# Point this at your deployment. Container port 8012 is published on host 8013.
BASE_URL = os.environ.get("GLINER_BASE_URL", "http://192.168.1.177:8013")

# Inference is bounded server-side by REQUEST_TIMEOUT_SECONDS (default 120),
# so a client timeout a bit above that is the sane default.
TIMEOUT = 130

session = requests.Session()
session.headers.update({"Content-Type": "application/json"})


def get(path):
    """GET a path and return parsed JSON."""
    r = session.get(f"{BASE_URL}{path}", timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


def post(path, payload):
    """POST JSON and return parsed JSON. Raises on non-2xx."""
    r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


def post_raw(path, payload):
    """POST JSON and return (status_code, parsed_body_or_text). Never raises."""
    r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text


def show(obj):
    print(json.dumps(obj, indent=2, ensure_ascii=False))


print("BASE_URL =", BASE_URL)

## 2. Health, deep health, and version

Three GET routes, and they answer different questions.

| Route | Runs inference? | Use it for |
|---|---|---|
| `GET /health` | no | Container `HEALTHCHECK`, cheap liveness, load level |
| `GET /health/deep` | **yes, always** | Monitoring. `503` when the model is not answering |
| `GET /version` | no | Provenance: library, model, and weight fingerprint |

Fields to read on `/health`:

- `loaded` — is the model resident? With `MODEL_PRELOAD=0` this stays `false`
  until the first inference request.
- `device` — `"cuda"` on a healthy Jetson. `"cpu"` means the GPU was not visible
  to the container; see the runbook.
- `architecture` / `model_class` — `span` + `GLiNER2` for the 2.x models,
  `boundary` + a boundary extractor class for 2.5. On this deployment
  `model_class` is `BoundaryExtractor`; `AutoExtractor` is the dispatcher, not
  the loaded class, so assert on `architecture` rather than on `model_class`.
- `inflight` / `saturated` — how many requests hold an inference slot, and
  whether the next one will queue.


In [ ]:
health = get("/health")
show(health)

assert health["status"] == "ok"
if health.get("device") != "cuda":
    print("\nWARNING: not running on GPU. On a Jetson this is a fault, not a mode.")

# Architecture decides which routes exist. Five of the twelve need "boundary".
ARCH = health.get("architecture")
IS_BOUNDARY = ARCH == "boundary"
print("\narchitecture :", ARCH)
print("boundary-only routes available:", IS_BOUNDARY)


In [ ]:
show(get("/version"))

# /health/deep always probes. It is a 503 when the model is not answering,
# so a bare status check is enough - no JSON parsing needed to alert on it.
r = session.get(f"{BASE_URL}/health/deep", timeout=TIMEOUT)
deep = r.json()
print("\nHTTP", r.status_code, "| status:", deep["status"])
print("probe:", deep.get("probe"))


`/version` carries provenance, not just a version string.
`model_revision` is a stable 16-hex-char fingerprint of the weights on disk —
stamp it onto extracted rows, otherwise extractions from two different
checkpoints are indistinguishable once stored.

`/health` is *liveness*, not proof the model works: a wedged worker keeps
answering `"status": "ok"` there indefinitely. `/health/deep` always runs a real
forward pass, and **deliberately bypasses the inference semaphore** so a wedged
worker stays distinguishable from a merely busy one. It returns `503` with
`"status": "degraded"` on failure, which makes `curl -f` a complete check.

Measured on this box: while a 48-document batch held the only inference slot,
`/health/deep` came back `200` in 1.56 s while a normal request queued 4.08 s.
Single measurement — indicative of the bypass working, not a latency SLO.


In [ ]:
smoke = post("/extract_entities", {
    "text": "Patient received 400mg ibuprofen for severe headache at 2 PM.",
    "labels": ["medication", "dosage", "symptom", "time"],
})
show(smoke)

# Verified against fastino/gliner2.5-base-v1 on jarvita-agx:
# {"entities": {"medication": ["ibuprofen"], "dosage": ["400mg"],
#               "symptom": ["severe headache"], "time": ["2 PM"]}}
#
# Note "severe headache", not "headache". Span boundaries are the model's
# choice and they shift between checkpoints - do not assume they match a term
# list you hold downstream.


## 3. Entity extraction — `POST /extract_entities`

Zero-shot NER. The label set travels with the request; nothing is fine-tuned,
and the same loaded weights answer any label set you send.

| Field | Type | Notes |
|---|---|---|
| `text` | string | non-empty, `<= MAX_TEXT_CHARS` (default 20000) |
| `labels` | list of strings, **or** object label → description | non-empty, `<= MAX_LABELS` (default 256) |

The response keys are exactly the labels you sent. A label with no match maps to
an empty list.

In [ ]:
clinical = post("/extract_entities", {
    "text": "Patient received 400mg ibuprofen for severe headache at 2 PM.",
    "labels": ["medication", "dosage", "symptom", "time"],
})
show(clinical)

The same model, a completely different domain and label set — no reload, no
fine-tune. This is the whole point of the zero-shot schema:

In [ ]:
business = post("/extract_entities", {
    "text": "Apple CEO Tim Cook announced record revenue in Cupertino.",
    "labels": ["company", "person", "location", "job_title"],
})
show(business)

### Steering labels with descriptions

Passing `labels` as an object maps each label to a natural-language description.
Use this when a bare label is ambiguous — `"reference"` could mean a citation, a
ticket number, or a job reference, and the description settles it.

In [ ]:
described = post("/extract_entities", {
    "text": "Patient received 400mg ibuprofen for severe headache at 2 PM.",
    "labels": {
        "medication": "name of a drug administered to the patient",
        "dosage": "amount and unit of the drug",
    },
})
show(described)

### Multilingual

`fastino/gliner2.5-multi-v1` is the multilingual checkpoint (mDeBERTa-v3-base
encoder). The cell below only produces meaningful results if the service is
running that `MODEL_ID` — the English-only models will do something, but not
something you should rely on.

In [ ]:
current_model = get("/health")["model_id"]
print("serving:", current_model)

if "multi" not in current_model:
    print("NOTE: not a multilingual checkpoint — treat the output below as a curiosity.")

show(post("/extract_entities", {
    "text": "Luca de Meo, PDG de Renault, a annoncé une nouvelle usine à Douai.",
    "labels": ["person", "company", "location"],
}))

Worth knowing: on a four-sentence comparison run on this box, `gliner2.5-multi`
returned `"PDG de Renault"` as the company for a sentence like this one, where
`gliner2-large` returned `"Renault"` — the 2.5 model swept the job title into
the span. **Four sentences is not an evaluation.** It is enough to justify one
habit: when you change `MODEL_ID`, re-check span boundaries against your own
texts before assuming downstream string matching still works.

## 4. Classification — `POST /classify_text`

| Field | Type | Notes |
|---|---|---|
| `text` | string | non-empty, `<= MAX_TEXT_CHARS` |
| `labels` | list of class labels, **or** object task name → class labels | non-empty, `<= MAX_LABELS` |

Prefer the object form. It names the task, so the response key is predictable
instead of something you have to discover.

In [ ]:
sentiment = post("/classify_text", {
    "text": "The battery life is terrible and it overheats constantly.",
    "labels": {"sentiment": ["positive", "negative", "neutral"]},
})
show(sentiment)

# Verified response from jarvita-agx: {"sentiment": "negative"}

Several independent classification tasks can ride along in one request — add
more keys to the `labels` object. One forward pass, several answers.

In [ ]:
show(post("/classify_text", {
    "text": "The battery life is terrible and it overheats constantly.",
    "labels": {
        "sentiment": ["positive", "negative", "neutral"],
        "topic": ["battery", "display", "software", "build quality"],
    },
}))

## 5. Structured extraction — `POST /extract_structured`

Declare a record schema, get records filled from free text.

Each top-level key of `schema` is a record name. Its value is a list of
`::`-delimited field specs:

```
"<field_name>::<dtype>"
"<field_name>::<dtype>::<description>"
```

`dtype` is `str` or `list`. The optional third segment steers extraction.

In [ ]:
product = post("/extract_structured", {
    "text": "The Sony WH-1000XM5 headphones cost $399 and ship in 3 days.",
    "schema": {"product": ["name::str", "price::str", "shipping::str"]},
})
show(product)

# Verified against fastino/gliner2.5-base-v1 on jarvita-agx:
# {"product": [{"name": "Sony WH-1000XM5", "price": "$399",
#               "shipping": "3 days"}]}


Note the value is a **list** of records, because a schema can match more than
once in a document. Only index `[0]` when you know the text holds a single
record.

In [ ]:
records = product["product"]
print(f"{len(records)} record(s)")
for i, rec in enumerate(records):
    print(f"  [{i}] {rec}")

With field descriptions, for a schema where bare names would be ambiguous:

In [ ]:
show(post("/extract_structured", {
    "text": "Goldman Sachs processed a $2.5M equity trade for Tesla Inc.",
    "schema": {
        "transaction": [
            "broker::str::Financial institution",
            "amount::str::Transaction amount",
            "security::str::Stock name",
        ]
    },
}))

## 6. Multi-task — `POST /extract_multitask`

Entities, a classification, and a structured record from **one** forward pass
over one text.

**Everything nests under `schema_config`.** There are no top-level `entities` /
`classification` / `structure` keys. Putting them at the top level is the most
common mistake with this endpoint and returns `400`.

| Field | Notes |
|---|---|
| `schema_config.entities` | list of entity labels |
| `schema_config.classification` | `{"name": str, "labels": [str, ...]}` |
| `schema_config.structure` | `{"name": str, "fields": [{"name", "dtype", "description", "choices"}, ...]}` |

All three sub-keys are optional; supply at least one.

In [ ]:
multitask = post("/extract_multitask", {
    "text": "Apple CEO Tim Cook announced record revenue in Cupertino.",
    "schema_config": {
        "entities": ["company", "person", "location"],
        "classification": {"name": "sentiment", "labels": ["positive", "negative"]},
        "structure": {
            "name": "announcement",
            "fields": [
                {"name": "who", "dtype": "str"},
                {"name": "what", "dtype": "str"},
            ],
        },
    },
})
show(multitask)

# Verified response from jarvita-agx:
# {"announcement": [{"who": "Tim Cook", "what": "record revenue"}],
#  "entities": {"company": ["Apple"], "person": ["Tim Cook"],
#               "location": ["Cupertino"]},
#  "sentiment": "positive"}

The response is flat: entities under `entities`, the classification under the
`name` you gave it, the structure under its `name` as a list of records. Key
ordering is not guaranteed — always address by key.

In [ ]:
print("entities      :", multitask["entities"])
print("sentiment     :", multitask["sentiment"])
print("announcement  :", multitask["announcement"][0])

### The wrong shape, for reference

This is what a top-level payload gets you. Run it once so the failure is
recognizable when it shows up in a client:

In [ ]:
status, body = post_raw("/extract_multitask", {
    "text": "Apple CEO Tim Cook announced record revenue in Cupertino.",
    "entities": ["company", "person", "location"],   # WRONG: must nest under schema_config
})
print("HTTP", status)
show(body)

### `choices`: constraining a field to a closed vocabulary

`structure.fields` entries accept `choices`, which restricts the field to a
fixed set rather than free extraction. Useful when the downstream consumer
expects an enum.

In [ ]:
show(post("/extract_multitask", {
    "text": "Support ticket: the checkout page returns a 500 error for all EU customers. "
            "This is blocking revenue and needs attention today.",
    "schema_config": {
        "entities": ["component", "error_code", "region"],
        "classification": {"name": "urgency", "labels": ["low", "medium", "high"]},
        "structure": {
            "name": "ticket",
            "fields": [
                {"name": "summary", "dtype": "str", "description": "one-line problem statement"},
                {"name": "area", "dtype": "str", "choices": ["frontend", "backend", "infrastructure"]},
            ],
        },
    },
}))

## 7. Error semantics

The service bounds everything and returns defined status codes. These are not
edge cases to ignore — `503` in particular is normal operation on a single-GPU
box, and `400` is now returned for things that used to be silently ignored.

| Status | Trigger |
|---|---|
| 400 | **Unknown key** in the payload or in `schema_config` — the detail lists the allowed keys |
| 400 | Malformed payload: missing/empty `text`/`texts`, bad `labels` type, missing `schema_config`, or a limit on label/field count |
| 400 | Out-of-range or wrong-typed inference option: `threshold` outside `[0, 1]`, non-boolean `include_spans`, non-positive `max_len` or `batch_size` |
| 413 | `text` longer than `MAX_TEXT_CHARS` (default 20000) |
| 413 | `texts` longer than `MAX_BATCH_SIZE` (default 64) |
| 413 | Batch character total over `MAX_BATCH_CHARS` (default 200000) |
| 501 | A boundary-only route called on a span checkpoint |
| 503 | No inference slot free within `INFERENCE_ACQUIRE_TIMEOUT_SECONDS` (default 10) |
| 503 | `/health/deep` probe failed or timed out — body carries `"status": "degraded"` |
| 504 | Inference exceeded `REQUEST_TIMEOUT_SECONDS` (default 120) |

Note the split: label and schema-field counts are `400`; anything measured in
characters or documents is `413`.

The strict key validation is the change most likely to break an existing
client. A key that used to be dropped now fails the request, which is the point
— a typo previously produced a plausible result that was quietly missing data.


In [ ]:
# 400 - missing text
print("--- missing 'text' ---")
print("HTTP %s | %s" % post_raw("/extract_entities", {"labels": ["person"]}))

# 400 - labels of the wrong type
print("\n--- 'labels' as a string ---")
print("HTTP %s | %s" % post_raw("/extract_entities", {"text": "Tim Cook.", "labels": "person"}))

# 400 - unknown payload key (typo). Used to be silently ignored.
print("\n--- typo'd inference option ---")
print("HTTP %s | %s" % post_raw(
    "/extract_entities", {"text": "Tim Cook.", "labels": ["person"], "treshold": 0.5}))

# 400 - unknown schema_config key
print("\n--- typo'd schema_config key ---")
print("HTTP %s | %s" % post_raw(
    "/extract_multitask", {"text": "Tim Cook.", "schema_config": {"entitys": ["person"]}}))

# 400 - task field at the top level instead of nested
print("\n--- 'entities' at the top level ---")
print("HTTP %s | %s" % post_raw(
    "/extract_multitask", {"text": "Tim Cook.", "entities": ["person"]}))

# 400 - threshold out of range
print("\n--- threshold 1.5 ---")
print("HTTP %s | %s" % post_raw(
    "/extract_entities", {"text": "Tim Cook.", "labels": ["person"], "threshold": 1.5}))

# 413 - text over MAX_TEXT_CHARS
print("\n--- oversized 'text' ---")
MAX_TEXT_CHARS = 20_000    # server default; raise here if your deployment overrides it
print("HTTP %s | %s" % post_raw(
    "/extract_entities", {"text": "a" * (MAX_TEXT_CHARS + 1), "labels": ["x"]}))


### Handling 503 correctly

`503` is backpressure. The server is telling you the GPU queue is full, which on
a box with `MAX_CONCURRENT_INFERENCES=1` is the designed behavior under load.
The correct client response is retry with backoff and jitter — **not** more
concurrent requests, which makes it worse.

In [ ]:
import random


def post_with_retry(path, payload, attempts=5, base_delay=0.5):
    """POST with exponential backoff + jitter on 503 (busy) and 504 (timeout)."""
    for attempt in range(attempts):
        r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)
        if r.status_code not in (503, 504):
            r.raise_for_status()
            return r.json()
        if attempt == attempts - 1:
            r.raise_for_status()
        delay = base_delay * (2 ** attempt) + random.uniform(0, base_delay)
        print(f"  HTTP {r.status_code}, retrying in {delay:.2f}s "
              f"(attempt {attempt + 1}/{attempts})")
        time.sleep(delay)


show(post_with_retry("/extract_entities", {
    "text": "Apple CEO Tim Cook announced record revenue in Cupertino.",
    "labels": ["company", "person", "location"],
}))

## 8. Throughput: sequential vs concurrent

Two things get measured below, and they are not the same thing:

- **Sequential HTTP** — 16 requests, one after another. This is what a naive
  bulk client does.
- **Concurrent HTTP** — 16 requests fired from a thread pool. With
  `MAX_CONCURRENT_INFERENCES=1` the server serializes them at the semaphore
  anyway, so this mostly measures how well overlapping request handling hides
  HTTP overhead — and it will start returning `503` once requests queue past
  `INFERENCE_ACQUIRE_TIMEOUT_SECONDS`.

Timings depend on the model, the document, and whatever else is running on the
box. Run this on your own deployment rather than trusting a number in a doc.

In [ ]:
DOC = "Apple CEO Tim Cook announced record revenue in Cupertino."
LABELS = ["company", "person", "location"]
N = 16

payload = {"text": DOC, "labels": LABELS}

# Warm up: with MODEL_PRELOAD=0 the first request pays the model load cost and
# would otherwise dominate the measurement.
post("/extract_entities", payload)
print("warm-up done")

In [ ]:
# --- Sequential ---
t0 = time.perf_counter()
for _ in range(N):
    post("/extract_entities", payload)
seq_total_ms = (time.perf_counter() - t0) * 1000

print(f"sequential : {seq_total_ms:8.1f} ms total   "
      f"{seq_total_ms / N:6.1f} ms/doc")

In [ ]:
# --- Concurrent over HTTP ---
def one_call(_):
    r = session.post(f"{BASE_URL}/extract_entities", json=payload, timeout=TIMEOUT)
    return r.status_code


t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as pool:
    statuses = list(pool.map(one_call, range(N)))
conc_total_ms = (time.perf_counter() - t0) * 1000

ok = sum(1 for s in statuses if s == 200)
busy = sum(1 for s in statuses if s == 503)

print(f"concurrent : {conc_total_ms:8.1f} ms total   "
      f"{conc_total_ms / N:6.1f} ms/doc")
print(f"             {ok} x 200, {busy} x 503 (busy)")
print(f"\nspeedup vs sequential: {seq_total_ms / conc_total_ms:.2f}x")

if busy:
    print("\n503s are expected here: the server has one inference slot and rejects "
          "anything that cannot get it within INFERENCE_ACQUIRE_TIMEOUT_SECONDS.")

### Reference: what batching actually buys

The numbers above are HTTP-level and both paths send one document per request.
The real lever is the **batch routes**, which take a `texts` list and hold a
single inference slot for the whole set. Measured on `jarvita-agx`:

| Comparison | Result |
|---|---|
| `gliner2.5-multi`, 16 documents, model-level batched vs sequential | 24.0 ms/doc vs 100.4 ms/doc — **4.2x** |
| `gliner2.5-base`, 32 documents, one `/extract_multitask_batch` vs 3 sequential single calls per doc | 25.8 ms/doc vs 274.9 ms/doc — **10.7x** |

Measured on a shared box, one run each — indicative of the order of magnitude,
not a controlled benchmark.

Section 12 reruns the second comparison against your own deployment.

Two things that were measured and did **not** help:

- **Schema caching between batch calls.** 127.41 ms versus 127.42 ms over 16
  documents. Noise — do not build it.
- **Raising `MAX_CONCURRENT_INFERENCES`.** The GPU serializes the work anyway;
  more concurrency multiplies peak activation memory without buying throughput.

Single-document latency by checkpoint on the same box (average of 5 runs, six
input sentences, shared box — indicative only):

| `MODEL_ID` | Avg latency | On disk |
|---|---|---|
| `fastino/gliner2.5-base-v1` | 83 ms | 748 MB |
| `fastino/gliner2.5-multi-v1` | 94 ms | 1.1 GB |
| `fastino/gliner2-large-v1` | 109 ms | 1.9 GB |

On that run `gliner2.5-multi` produced duplicate, garbled records on structured
extraction and under-recalled English entities; `gliner2.5-base` did neither.
**Six sentences is not an evaluation** — it is a reason to re-check outputs
against your own texts whenever you change `MODEL_ID`.


In [ ]:
# Per-endpoint latency on your deployment, for comparison with the table above.
cases = [
    ("extract_entities  (clinical)", "/extract_entities", {
        "text": "Patient received 400mg ibuprofen for severe headache at 2 PM.",
        "labels": ["medication", "dosage", "symptom", "time"],
    }),
    ("extract_entities  (business)", "/extract_entities", {
        "text": "Apple CEO Tim Cook announced record revenue in Cupertino.",
        "labels": ["company", "person", "location"],
    }),
    ("classify_text", "/classify_text", {
        "text": "The battery life is terrible and it overheats constantly.",
        "labels": {"sentiment": ["positive", "negative", "neutral"]},
    }),
    ("extract_structured", "/extract_structured", {
        "text": "The Sony WH-1000XM5 headphones cost $399 and ship in 3 days.",
        "schema": {"product": ["name::str", "price::str", "shipping::str"]},
    }),
]

RUNS = 5
print(f"{'endpoint':<32} {'mean ms':>9} {'min ms':>9} {'max ms':>9}")
print("-" * 62)
for name, path, body in cases:
    post(path, body)  # warm
    samples = []
    for _ in range(RUNS):
        t = time.perf_counter()
        post(path, body)
        samples.append((time.perf_counter() - t) * 1000)
    print(f"{name:<32} {sum(samples)/len(samples):9.1f} "
          f"{min(samples):9.1f} {max(samples):9.1f}")

print(f"\nmodel: {get('/health')['model_id']}")
print("Shared box - these are indicative, not a controlled benchmark.")

## 9. Inference options

Every `POST` extract/classify route — single and batch — accepts the same
optional parameters at the **top level** of the payload, alongside
`text`/`texts` and the task fields.

| Key | Type | Default | Effect |
|---|---|---|---|
| `threshold` | number in `[0, 1]` | `0.5` | Minimum score for a prediction to survive |
| `include_confidence` | bool | `false` | Add per-prediction scores; **changes response shape** |
| `include_spans` | bool | `false` | Add real character offsets; **changes response shape** |
| `max_len` | positive int | model default | Chunk window for long documents |
| `overlap_policy` | string | model default | How overlapping chunk predictions are reconciled |
| `batch_size` | positive int | model default | Batch routes only; clamped to `MAX_BATCH_SIZE` |

These used to be accepted and silently dropped. They are honoured now, and bad
values are a `400` rather than a shrug.


### `threshold` genuinely filters

The default `0.5` is permissive for a confident model. On clean text this
checkpoint sits above `0.99`, so nothing between `0.2` and `0.9` changes the
answer — `threshold` is not a useful precision dial until you are near the top
of the range. The sweep below shows where it starts biting.

Note the labels never disappear: a filtered-out label stays as a key with an
empty list.


In [ ]:
AMBIGUOUS = ("The Jaguar was spotted near the old mill road at dusk "
             "by a ranger from Devon.")
AMB_LABELS = ["company", "animal", "person", "location"]

# What the model actually believes:
show(post("/extract_entities", {
    "text": AMBIGUOUS, "labels": AMB_LABELS, "include_confidence": True,
}))

print("\n--- threshold sweep ---")
for th in (0.2, 0.9, 0.98, 0.995, 0.999):
    r = post("/extract_entities", {"text": AMBIGUOUS, "labels": AMB_LABELS, "threshold": th})
    kept = {k: v for k, v in r["entities"].items() if v}
    print(f"  threshold={th:<6} kept: {kept}")

# Verified on fastino/gliner2.5-base-v1 (jarvita-agx):
#   0.2   -> animal Jaguar | person ranger | location old mill road, Devon
#   0.9   -> identical to 0.2
#   0.98  -> animal Jaguar | location old mill road
#   0.995 -> animal Jaguar
#   0.999 -> nothing


In [ ]:
# Out-of-range and wrong-typed values are rejected, not ignored.
for bad in ({"threshold": 1.5},
            {"threshold": "high"},
            {"include_spans": "yes"},
            {"max_len": 0}):
    payload = {"text": "Tim Cook.", "labels": ["person"], **bad}
    print("%-28s HTTP %s | %s" % (str(bad), *post_raw("/extract_entities", payload)))


### `include_confidence` and `include_spans` change the response shape

This is the part that breaks clients. Without the flags an entity is a bare
string; with either flag it becomes an object. Decide once per consumer and
stay consistent — a client that flips these mid-stream writes two incompatible
shapes into the same dataset.

`start`/`end` are real character offsets: `text[start:end]` reproduces the span
exactly. The assertion below checks that against the live service rather than
taking the doc's word for it.


In [ ]:
SPAN_TEXT = "Apple CEO Tim Cook announced iPhone 15 in Cupertino."
SPAN_LABELS = ["company", "person", "product", "location"]

base = post("/extract_entities", {"text": SPAN_TEXT, "labels": SPAN_LABELS})
conf = post("/extract_entities", {"text": SPAN_TEXT, "labels": SPAN_LABELS,
                                  "include_confidence": True})
spans = post("/extract_entities", {"text": SPAN_TEXT, "labels": SPAN_LABELS,
                                   "include_spans": True})
both = post("/extract_entities", {"text": SPAN_TEXT, "labels": SPAN_LABELS,
                                  "include_confidence": True, "include_spans": True})

print("--- bare ---");                show(base)
print("\n--- include_confidence ---"); show(conf)
print("\n--- include_spans ---");      show(spans)
print("\n--- both ---");               show(both)

# Offsets are real: verify rather than trust.
for label, items in spans["entities"].items():
    for item in items:
        sliced = SPAN_TEXT[item["start"]:item["end"]]
        assert sliced == item["text"], (label, sliced, item)
print("\nall offsets round-trip against the source text")


In [ ]:
# Classification changes shape too: a bare label becomes {label, confidence}.
print("--- bare ---")
show(post("/classify_text", {
    "text": "The battery life is terrible and it overheats constantly.",
    "labels": {"sentiment": ["positive", "negative", "neutral"]},
}))

print("\n--- include_confidence ---")
show(post("/classify_text", {
    "text": "The battery life is terrible and it overheats constantly.",
    "labels": {"sentiment": ["positive", "negative", "neutral"]},
    "include_confidence": True,
}))

# Verified: {"sentiment": "negative"} becomes
#           {"sentiment": {"label": "negative", "confidence": 0.9999998807907104}}


### `max_len` and `overlap_policy`

Chunking controls for documents longer than the model window. Both are
forwarded to `gliner2` unchanged — the accepted `overlap_policy` values are the
library's, and the API only checks that it is a non-empty string.

Leave both unset unless you are actually chunking. On short documents they
change nothing and only add a way to get it wrong.


In [ ]:
show(post("/extract_entities", {
    "text": ("Apple CEO Tim Cook announced iPhone 15 in Cupertino. "
             "Satya Nadella leads Microsoft from Redmond."),
    "labels": ["company", "person", "location"],
    "max_len": 384,
    "overlap_policy": "longest",
}))


## 10. Relation extraction

GLiNER2.5 extracts **typed relations**: given a text and a set of relation
names, it returns `[head, tail]` pairs. Two ways in:

1. `POST /extract_relations` — the dedicated route.
2. `schema_config.relations` on `/extract_multitask` — relations in the same
   forward pass as entities, classification, and structure. This used to be
   accepted and silently dropped; it is honoured now.

Both need a **boundary** checkpoint. On a span model they return `501` naming
the architecture actually loaded, rather than failing obscurely.

The model applies no type constraint you did not give it, and it will happily
return arguments of the wrong type. **Validate relation output downstream.**


In [ ]:
RELATION_TEXT = "Satya Nadella, CEO of Microsoft, met Sam Altman of OpenAI in Seattle."
RELATIONS = ["works_for", "met_with", "located_in"]

status, body = post_raw("/extract_relations", {"text": RELATION_TEXT, "relations": RELATIONS})
print("HTTP", status)
show(body)

if status == 501:
    print("\nExpected on a span checkpoint. Set MODEL_ID to a GLiNER2.5 model.")

# Verified on fastino/gliner2.5-base-v1 (jarvita-agx):
# {"relation_extraction": {
#     "works_for":  [["Satya Nadella", "Microsoft"]],
#     "met_with":   [["Satya Nadella", "Sam Altman"]],
#     "located_in": [["Sam Altman", "Seattle"]]}}
#
# located_in returned a PERSON, not the organization you might expect.


In [ ]:
# include_confidence changes the relation shape most: each [head, tail] pair
# becomes {"head": {...}, "tail": {...}}. Head and tail carry the SAME score -
# it is the confidence of the relation, not of each argument.
if IS_BOUNDARY:
    show(post("/extract_relations", {
        "text": RELATION_TEXT,
        "relations": ["works_for", "met_with"],
        "include_confidence": True,
    }))
else:
    print("skipped: needs a boundary checkpoint")


In [ ]:
# Relations alongside entities, in ONE forward pass, via schema_config.
status, body = post_raw("/extract_multitask", {
    "text": RELATION_TEXT,
    "schema_config": {
        "entities": ["company", "person"],
        "relations": ["works_for", "met_with"],
    },
})
print("HTTP", status)
show(body)

# Verified: entities under "entities", relations under "relation_extraction".
# schema_config accepts exactly four keys - entities, classification,
# relations, structure - and anything else is a 400.


## 11. The four batch routes

`POST /extract_entities_batch`, `/classify_text_batch`,
`/extract_relations_batch`, `/extract_multitask_batch`.

All four:

- require a **boundary** checkpoint (`501` otherwise);
- return `{"results": [...]}` **positionally aligned** with the `texts` sent;
- hold **one** inference slot for the whole batch, rather than contending for
  the semaphore once per document;
- accept every inference option from section 9, plus `batch_size`;
- are bounded by `MAX_BATCH_SIZE` (64 documents) **and** `MAX_BATCH_CHARS`
  (200000 characters summed across the batch). Either overrun is a `413`.

Both bounds exist because a document count alone does not bound memory: 64
documents of `MAX_TEXT_CHARS` each is what OOMed the box.


In [ ]:
BATCH_TEXTS = [
    "Apple CEO Tim Cook announced iPhone 15 in Cupertino.",
    "Satya Nadella leads Microsoft from Redmond.",
]

if not IS_BOUNDARY:
    print("skipping section 11: needs a GLiNER2.5 boundary checkpoint")
else:
    print("--- extract_entities_batch ---")
    show(post("/extract_entities_batch",
              {"texts": BATCH_TEXTS, "labels": ["company", "person", "location"]}))

    print("\n--- classify_text_batch ---")
    show(post("/classify_text_batch", {
        "texts": ["The battery is terrible.", "Works flawlessly, love it."],
        "labels": {"sentiment": ["positive", "negative", "neutral"]},
    }))

    print("\n--- extract_relations_batch ---")
    show(post("/extract_relations_batch", {
        "texts": ["Satya Nadella, CEO of Microsoft, met Sam Altman of OpenAI.",
                  "Tim Cook runs Apple from Cupertino."],
        "relations": ["works_for", "met_with"],
    }))

    print("\n--- extract_multitask_batch ---")
    show(post("/extract_multitask_batch", {
        "texts": BATCH_TEXTS,
        "schema_config": {
            "entities": ["company", "person"],
            "classification": {"name": "sentiment", "labels": ["positive", "negative"]},
        },
    }))


Verified `extract_relations_batch` output on `gliner2.5-base-v1`:

```json
{"results": [
  {"relation_extraction": {"works_for": [["Satya Nadella", "Microsoft"]],
                           "met_with": [["Satya Nadella", "Sam Altman"]]}},
  {"relation_extraction": {"works_for": [["Tim Cook", "Cupertino"]],
                           "met_with": []}}
]}
```

Read the second result closely: `works_for` returned
`["Tim Cook", "Cupertino"]` — a place, not the company in the same sentence.
Relation output is not type-checked for you.

`include_spans` on a batch route gives offsets **relative to each document**,
not to the concatenated batch:


In [ ]:
if IS_BOUNDARY:
    spans_batch = post("/extract_entities_batch", {
        "texts": BATCH_TEXTS,
        "labels": ["company", "person"],
        "include_spans": True,
        "batch_size": 8,
    })
    show(spans_batch)

    # Verify: each document's offsets index into that document.
    for doc, result in zip(BATCH_TEXTS, spans_batch["results"]):
        for label, items in result["entities"].items():
            for item in items:
                assert doc[item["start"]:item["end"]] == item["text"], (doc, item)
    print("\nper-document offsets verified")
else:
    print("skipped: needs a boundary checkpoint")


### The two batch bounds

`MAX_BATCH_SIZE` counts documents; `MAX_BATCH_CHARS` counts characters. A batch
can sit well inside the document cap and still be rejected on characters. Chunk
clients against **both**.

Code defaults are 64 documents and 200000 characters, but both are environment
variables and a given deployment may run tighter. The cell below **discovers**
the live values from the `413` detail rather than assuming, then defines the
`chunk_batches` helper that section 12 uses.


In [ ]:
import re

# The two bounds are configurable per deployment, so discover them rather than
# hardcoding: a deliberate overrun reports the live value in the 413 detail.
MAX_BATCH_SIZE = 64        # code defaults, used if discovery is skipped
MAX_BATCH_CHARS = 200_000


def _limit_from(detail, key, fallback):
    m = re.search(rf"{key}=(\d+)", str(detail))
    return int(m.group(1)) if m else fallback


if IS_BOUNDARY:
    print("--- too many documents ---")
    status, body = post_raw(
        "/extract_entities_batch",
        {"texts": ["hello world"] * (MAX_BATCH_SIZE + 1), "labels": ["person"]})
    print("HTTP", status, "|", body)
    MAX_BATCH_SIZE = _limit_from(body.get("detail"), "MAX_BATCH_SIZE", MAX_BATCH_SIZE)

    print("\n--- too many characters (only 15 documents) ---")
    status, body = post_raw(
        "/extract_entities_batch",
        {"texts": ["a" * 19_000] * 15, "labels": ["person"]})
    print("HTTP", status, "|", body)
    MAX_BATCH_CHARS = _limit_from(body.get("detail"), "MAX_BATCH_CHARS", MAX_BATCH_CHARS)

    print(f"\nlive bounds: MAX_BATCH_SIZE={MAX_BATCH_SIZE} "
          f"MAX_BATCH_CHARS={MAX_BATCH_CHARS}")
else:
    print("skipped: needs a boundary checkpoint")


def chunk_batches(texts, max_size=None, max_chars=None):
    """Split texts into batches respecting BOTH server bounds."""
    max_size = MAX_BATCH_SIZE if max_size is None else max_size
    max_chars = MAX_BATCH_CHARS if max_chars is None else max_chars
    batch, chars = [], 0
    for t in texts:
        if batch and (len(batch) >= max_size or chars + len(t) > max_chars):
            yield batch
            batch, chars = [], 0
        batch.append(t)
        chars += len(t)
    if batch:
        yield batch


sizes = [len(b) for b in chunk_batches(["a" * 19_000] * 40)]
print("\n40 x 19k-char documents chunk into batches of:", sizes)


## 12. Batched vs sequential, measured on your deployment

The comparison that matters for bulk work: **three single-document calls per
document** (entities, then classification, then structure) against **one
`/extract_multitask_batch` call** covering the whole set.

Reference numbers measured on `jarvita-agx` over 32 documents against
`fastino/gliner2.5-base-v1`:

| Approach | Per document | Documents/min |
|---|---|---|
| 3 sequential single calls per document | 274.9 ms | 218 |
| 1 `/extract_multitask_batch` call | 25.8 ms | 2326 |

**10.7x.** One run on a shared box — indicative of the order of magnitude, not
a controlled benchmark. Your run below will differ; expect the same shape.

The gain has two independent sources, worth keeping apart:

1. **Batching** amortizes per-call and per-request overhead across documents.
2. **Multitask** does all three tasks in one forward pass instead of three.

Calling `/extract_entities_batch`, `/classify_text_batch` and
`/extract_structured` separately buys (1) but not (2).


In [ ]:
N_DOCS = 32

CORPUS = [
    f"Apple CEO Tim Cook announced record revenue in Cupertino in quarter {i}."
    for i in range(N_DOCS)
]

ENTITY_LABELS = ["company", "person", "location"]
CLS_LABELS = {"sentiment": ["positive", "negative"]}
STRUCT_SCHEMA = {"announcement": ["who::str", "what::str"]}

MULTITASK_SCHEMA = {
    "entities": ENTITY_LABELS,
    "classification": {"name": "sentiment", "labels": ["positive", "negative"]},
    "structure": {
        "name": "announcement",
        "fields": [{"name": "who", "dtype": "str"}, {"name": "what", "dtype": "str"}],
    },
}

# Warm up so a cold load does not land inside a measurement.
post("/extract_entities", {"text": CORPUS[0], "labels": ENTITY_LABELS})
print(f"warm-up done | {N_DOCS} documents | model:", get("/health")["model_id"])


In [ ]:
# --- Sequential: three single-document calls per document ---
t0 = time.perf_counter()
for doc in CORPUS:
    post("/extract_entities", {"text": doc, "labels": ENTITY_LABELS})
    post("/classify_text", {"text": doc, "labels": CLS_LABELS})
    post("/extract_structured", {"text": doc, "schema": STRUCT_SCHEMA})
seq_ms = (time.perf_counter() - t0) * 1000

print(f"sequential (3 calls/doc) : {seq_ms:9.1f} ms total   "
      f"{seq_ms / N_DOCS:7.1f} ms/doc   {60_000 / (seq_ms / N_DOCS):7.0f} docs/min")


In [ ]:
# --- Batched: one multitask call, chunked against both server bounds ---
if not IS_BOUNDARY:
    print("skipped: /extract_multitask_batch needs a boundary checkpoint")
else:
    batches = list(chunk_batches(CORPUS))
    t0 = time.perf_counter()
    batched_results = []
    for batch in batches:
        r = post("/extract_multitask_batch",
                 {"texts": batch, "schema_config": MULTITASK_SCHEMA})
        batched_results.extend(r["results"])
    batch_ms = (time.perf_counter() - t0) * 1000

    assert len(batched_results) == N_DOCS, "results must align 1:1 with texts"

    print(f"multitask_batch          : {batch_ms:9.1f} ms total   "
          f"{batch_ms / N_DOCS:7.1f} ms/doc   {60_000 / (batch_ms / N_DOCS):7.0f} docs/min")
    print(f"\nspeedup: {seq_ms / batch_ms:.1f}x  "
          f"({len(batches)} HTTP request(s) instead of {N_DOCS * 3})")
    print("\nShared box, one run - indicative, not a controlled benchmark.")
    print("\nfirst result:")
    show(batched_results[0])


### What not to try

- **Schema caching between batch calls.** Measured at 127.41 ms versus
  127.42 ms over 16 documents — a 0.01 ms difference, which is noise. It is not
  an optimization.
- **Raising `MAX_CONCURRENT_INFERENCES`.** The GPU serializes the work anyway.
  More concurrency multiplies peak activation memory and, on a box sharing
  unified memory with an LLM, takes it from something else.
- **Bigger batches without watching memory.** `MAX_BATCH_CHARS` is the bound
  that actually holds; raising it is the OOM lever. Watch `docker stats`
  through a full-size batch before trusting a new value.


## Where to go next

| Need | Document |
|---|---|
| All 12 endpoints, inference options, error semantics | [`api.md`](api.md) |
| Deploy, `/health/deep` monitoring, env vars, batch sizing, rollback, failures | [`runbook.md`](runbook.md) |
| What GLiNER2/2.5 is, boundary-only capabilities, model comparison | [`wiki.md`](wiki.md) |
| Jetson build decisions and the `sbsa/cu130` trap | [`../JETSON.md`](../JETSON.md) |

Live API schema, straight from the running service:

```
http://192.168.1.177:8013/docs
http://192.168.1.177:8013/redoc
http://192.168.1.177:8013/openapi.json
```

List every route the running build actually exposes:

```bash
curl -s http://192.168.1.177:8013/openapi.json \
  | python3 -c "import json,sys; d=json.load(sys.stdin); [print(m.upper(), p) for p,v in d['paths'].items() for m in v]"
```
